# Lightweight Local RG Pipeline Demo

This notebook runs the full in-memory retrieval-generation pipeline with no API keys.

In [ ]:
from CUNY_course.example_code.rag_pipeline.data_prep_step import prepare_documents
from CUNY_course.example_code.rag_pipeline.chunking_step import chunk_documents
from CUNY_course.example_code.rag_pipeline.embedding_step import build_embedding_index
from CUNY_course.example_code.rag_pipeline.metadata_step import enrich_chunks_with_metadata
from CUNY_course.example_code.rag_pipeline.retrieval_step import retrieve_with_rerank

from dotenv import load_dotenv
import os
from openai import AzureOpenAI
from IPython.display import HTML, display

# Enable soft wrapping for long stdout/text outputs in notebook cells.
display(HTML("""<style>pre { white-space: pre-wrap !important; word-break: break-word !important; }</style>"""))

## Step 1: Data prep

In [ ]:
documents = prepare_documents()
len(documents), documents[0].doc_id, documents[0].title

## Step 2: Chunking

In [ ]:
chunks = chunk_documents(documents, chunk_size_chars=200, overlap_chars=50)
print( len(chunks))

chunks

## Step 3: Embedding + in-memory FAISS index

In [ ]:
embedding_index, chunk_lookup = build_embedding_index(chunks)
embedding_index.embedding_dim, len(embedding_index.chunk_ids)

## Step 4: Metadata

In [ ]:
chunks = enrich_chunks_with_metadata(chunks)
[(c.chunk_id, c.metadata.get('topic')) for c in chunks[:5]]

## Step 5: Retrieval + reranking

In [ ]:
import textwrap

query = "Can I bring my pet?"
hits = retrieve_with_rerank(
    query=query,
    embedding_index=embedding_index,
    chunk_lookup=chunk_lookup,
    top_k=5,
    candidate_k=8,
)
[(h.chunk.chunk_id, round(h.semantic_score, 3), round(h.lexical_score, 3), round(h.rerank_score, 3)) for h in hits]

for h in hits:
    print(f"--- {h.chunk.chunk_id} ---")
    print(textwrap.fill(h.chunk.text, width=100))
    print(h.chunk.metadata)
    print(h.semantic_score, h.lexical_score, h.rerank_score)
    print()

## Step 6: Generation

In [ ]:
load_dotenv()

client = AzureOpenAI(
    api_version="2025-03-01-preview",
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    timeout=30,
    max_retries=3,
 )


def build_prompt_local(query, hits):
    context_lines = [f"[{h.chunk.chunk_id}] {h.chunk.text}" for h in hits]
    context_block = "\n".join(context_lines)
    return (
        "Use only the context to answer the question in 1-3 complete sentences. "
        "If the context is insufficient, say: 'I don't know based on the provided context.' "
        "Do not include citation IDs in your response.\n\n"
        f"Question: {query}\n\n"
        f"Context:\n{context_block}\n\n"
        "Answer:"
    )


def _extract_text_from_response_output(response):
    chunks = []
    for out in getattr(response, "output", []) or []:
        for item in (getattr(out, "content", []) or []):
            text = getattr(item, "text", None)
            if text:
                chunks.append(text)
    return "\n".join(chunks).strip()


def generate_with_azure(prompt):
    response = client.responses.create(
        model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        input=[{"role": "user", "content": prompt}],
        reasoning={"effort": "minimal"},
        max_output_tokens=400,
    )

    text = (response.output_text or "").strip()
    if not text:
        text = _extract_text_from_response_output(response)
    if text:
        return text

    # Fallback only when the deployment returns no text from Responses API.
    chat = client.chat.completions.create(
        model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        messages=[
            {"role": "system", "content": "You are a concise, factual assistant."},
            {"role": "user", "content": prompt},
        ],
        max_completion_tokens=400,
    )
    return (chat.choices[0].message.content or "").strip()


prompt = build_prompt_local(query=query, hits=hits)
print(prompt)

In [ ]:
citations = [h.chunk.chunk_id for h in hits]
llm_answer = generate_with_azure(prompt)
final_answer = llm_answer.strip()

if citations:
    final_answer = f"{final_answer} [{' '.join(citations)}]"

final_answer, citations

In [ ]:
final_answer

In [ ]:
import textwrap

print("Answer:")
print(textwrap.fill(final_answer, width=100))

print("\nCitations:")
for citation in citations:
    print(f"- {citation}")

## Step 7: Interactive Playground

Use the controls below to test different queries and retrieval/chunking settings without editing code cells each time.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
import textwrap

query_widget = widgets.Text(
    value="Can I bring my pet?",
    description="Query:",
    layout=widgets.Layout(width="95%"),
)
chunk_size_widget = widgets.IntSlider(
    value=200, min=80, max=800, step=20, description="Chunk chars:"
)
overlap_widget = widgets.IntSlider(
    value=50, min=0, max=300, step=10, description="Overlap:"
)
top_k_widget = widgets.IntSlider(
    value=5, min=1, max=10, step=1, description="Top K:"
)
candidate_k_widget = widgets.IntSlider(
    value=8, min=2, max=20, step=1, description="Candidate K:"
)
run_button = widgets.Button(description="Run RAG", button_style="primary")
output = widgets.Output()

def run_interactive(_):
    with output:
        clear_output(wait=True)

        chunk_size = chunk_size_widget.value
        overlap = overlap_widget.value
        if overlap >= chunk_size:
            print("Overlap must be smaller than chunk size.")
            return

        local_chunks = chunk_documents(
            documents,
            strategy="character",
            chunk_size_chars=chunk_size,
            overlap_chars=overlap,
        )
        local_chunks = enrich_chunks_with_metadata(local_chunks)
        local_index, local_lookup = build_embedding_index(local_chunks)

        local_hits = retrieve_with_rerank(
            query=query_widget.value,
            embedding_index=local_index,
            chunk_lookup=local_lookup,
            top_k=top_k_widget.value,
            candidate_k=candidate_k_widget.value,
        )

        prompt_local = build_prompt_local(query_widget.value, local_hits)
        answer = generate_with_azure(prompt_local)
        local_citations = [h.chunk.chunk_id for h in local_hits]

        print(f"Chunks created: {len(local_chunks)}")
        print(f"Retrieved hits: {len(local_hits)}")
        print()
        print("Answer:")
        print(textwrap.fill(answer, width=100))
        print()
        print("Citations:")
        for c in local_citations:
            print(f"- {c}")

        print()
        print("Top hit previews:")
        for h in local_hits[:3]:
            print(f"--- {h.chunk.chunk_id} ---")
            print(textwrap.fill(h.chunk.text, width=100))
            print()

run_button.on_click(run_interactive)

display(
    widgets.VBox([
        query_widget,
        widgets.HBox([chunk_size_widget, overlap_widget]),
        widgets.HBox([top_k_widget, candidate_k_widget]),
        run_button,
        output,
    ])
)